# 03. Tide-corrected shoreline change using every clear scene

1. Downloads every Landsat 5/7/8/9 and Sentinel-2 scene that is at least 90% cloud-free over the study box (Earth Engine `computePixels`, no Drive export queue).
2. Downloads the EOT20 global tide model (about 2 GB, open access, SEANOE doi:10.17882/79489) to Colab's temporary disk.
3. Checks the tide model against the 1992 hourly Penang tide gauge record (UHSLC data via PANGAEA).
4. Extracts shorelines on the same transects as notebook 02, computes tide at each image time, estimates beach slope from the imagery, and reports raw, tide-corrected and tide-filtered trends plus a slope sensitivity test.

Needs `results/transects_utm47n.geojson` from notebook 02 (a copy is in `data/` of this repository) and, optionally, `vegetation_line_by_transect.csv` from the Google Earth step (also in `data/`).

Run time: cell 2 about 20 to 30 min; cell 3 about 5 to 10 min. Cell 2 can be re-run after a disconnection and continues where it stopped.

In [ ]:
!pip install -q rasterio scikit-image shapely eo-tides
from google.colab import drive
drive.mount('/content/drive')

# Get the code: clone this repository into Colab (edit REPO_URL if you forked it)
import os, sys
REPO_URL = 'https://github.com/nroselnik/kerachut-shoreline.git'
if not os.path.exists('/content/kerachut-shoreline'):
    os.system(f'git clone -q {REPO_URL} /content/kerachut-shoreline')
sys.path.insert(0, '/content/kerachut-shoreline/src')

import ee
# >>> Replace with YOUR Google Cloud project ID (registered for Earth Engine).
# >>> See docs/GEE_tutorial.md, section 3, for how to find it.
PROJECT = 'YOUR-GEE-PROJECT-ID'
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
import kerachut_tides_run as run

FOLDER = '/content/drive/MyDrive/Kerachut_GEE'
scenes = run.fetch_all_scenes(ee, FOLDER + '/all_scenes', min_clear=0.90)
tide_dir = run.setup_eot20()
print('downloads done')

In [ ]:
import kerachut_tides as kt, pandas as pd
REPO = '/content/kerachut-shoreline'

# 1. Tide model vs Penang tide gauge, 1992 (gauge at about 5.421 N, 100.347 E)
g = run.gauge_1992()
if g is not None:
    v, _ = kt.validate_with_gauge(g, 100.347, 5.421, tide_dir)
    print('Tide model vs gauge:', {a: round(b, 3) for a, b in v.items()})

# 2. Shoreline positions for every scene, fixed transects
P, Q, trs = kt.extract(FOLDER + '/all_scenes', REPO + '/data/transects_utm47n.geojson',
                       times_csv=FOLDER + '/all_scenes/_scene_times.csv')
print(Q.status.value_counts().to_string())

# 3. Tides, slope, corrected trends, figures 6 to 9
tide = kt.tides_eot20(P['date'], run.TIDE_LON, run.TIDE_LAT, tide_dir)
o = kt.analyse(P, tide, trs, FOLDER + '/results_tide', veg_csv=REPO + '/data/vegetation_line_by_transect.csv')
print(o['S'][['n', 'tanb', 'tanb_lo', 'tanb_hi', 'r']].round(3).to_string())
RZ = o['RZ']
print(RZ[RZ.period.isin(['1988-2017', '2018-2026'])][['treatment', 'period', 'series', 'n', 'LRR', 'CI95', 'p']].round(2).to_string())